# Qwen — RL (GRPO) với Learned Reward Model

**Yêu cầu:** Chạy `Reward_Model_Training.ipynb` trước để có checkpoint tại `mcs_train_content_model_outputs/reward_model/`.

**Điểm khác so với v1 (rule-based):**
- Reward không dùng heuristic tay — dùng regression model đã học từ engagement thật.
- Prompt vẫn là câu đầu tiên của bài thực (prefix completion).

**Vấn đề đã biết:** Reward model train trên 500 mẫu dining review — khả năng generalize ra ngoài domain này bị hạn chế. Xem phần 5 để đánh giá sau train bằng LLM judge.

## 0. Cài đặt

In [ ]:
# %pip install -U trl transformers accelerate peft bitsandbytes datasets pandas

## 1. Config

In [ ]:
import os, re, sys
import random
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from unittest.mock import MagicMock

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

REPO_ROOT = Path.cwd().resolve()
CSV_PATH  = REPO_ROOT / "dataset" / "500 posts diningreview.csv"

POLICY_MODEL_ID  = "Qwen/Qwen3.5-2B"   # đổi: 4B / 9B
REWARD_MODEL_DIR = REPO_ROOT / "mcs_train_content_model_outputs" / "reward_model"

OUTPUT_DIR = (
    REPO_ROOT / "mcs_train_content_model_outputs"
    / (POLICY_MODEL_ID.replace("/", "-") + "_rl_grpo_v2")
)
GRPO_DIR = str(OUTPUT_DIR / "grpo_policy")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_SEQ_LENGTH = 1024
LOAD_IN_4BIT   = True

assert CSV_PATH.is_file(),          f"Dataset not found: {CSV_PATH}"
assert REWARD_MODEL_DIR.exists(),   f"Reward model not found: {REWARD_MODEL_DIR} — chạy Reward_Model_Training.ipynb trước"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device          :", device)
print("policy model    :", POLICY_MODEL_ID)
print("reward model    :", REWARD_MODEL_DIR)
print("output dir      :", OUTPUT_DIR)

## 2. Load reward model

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer as RMTokenizer

rm_tokenizer = RMTokenizer.from_pretrained(str(REWARD_MODEL_DIR))
rm_model = AutoModelForSequenceClassification.from_pretrained(
    str(REWARD_MODEL_DIR),
    num_labels=1,
)
rm_model.eval()
rm_model.to(device)

print("Reward model loaded từ:", REWARD_MODEL_DIR)

@torch.no_grad()
def reward_fn(completions, **kwargs):
    """Dùng reward model để score mỗi completion. Trả về list[float]."""
    inputs = rm_tokenizer(
        completions,
        truncation=True,
        max_length=512,
        padding=True,
        return_tensors="pt",
    ).to(device)
    scores = rm_model(**inputs).logits.squeeze(-1)
    # Clamp về [0, 1] vì reward model predict trong khoảng này
    scores = scores.clamp(0.0, 1.0)
    return scores.tolist()

# Sanity check
test_scores = reward_fn(["Quán ngon lắm mọi người ơi 😍", "The food was acceptable."])
print("Test scores:", test_scores)

## 3. Load policy model (QLoRA) & chuẩn bị dataset

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset

bnb_config = BitsAndBytesConfig(
    load_in_4bit=LOAD_IN_4BIT,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    bnb_4bit_use_double_quant=True,
)

policy = AutoModelForCausalLM.from_pretrained(
    POLICY_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)
tok = AutoTokenizer.from_pretrained(POLICY_MODEL_ID, trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

policy = prepare_model_for_kbit_training(policy, use_gradient_checkpointing=True)

peft_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
policy = get_peft_model(policy, peft_cfg)
policy.print_trainable_parameters()

if not hasattr(policy, "warnings_issued"):
    policy.__dict__["warnings_issued"] = {}

In [ ]:
# Dataset: chỉ dùng câu đầu tiên làm prompt
df = pd.read_csv(str(CSV_PATH), encoding="utf-8")
df["text"] = df["text"].astype(str)
df = df[df["text"].str.len() > 50].copy()

def extract_first_sentence(text, min_len=15, max_len=200):
    match = re.search(r'(?<=[^\d])[.!?…]', text[min_len:])
    if match:
        return text[:min_len + match.start() + 1].strip()
    return text[:max_len].strip()

def build_grpo_dataset(frame, n=256, seed=SEED):
    sample = frame.sample(min(n, len(frame)), replace=n > len(frame), random_state=seed)
    rows = [{"prompt": extract_first_sentence(r["text"])} for _, r in sample.iterrows()]
    return Dataset.from_list(rows[:n])

grpo_train = build_grpo_dataset(df, n=256)
grpo_eval  = build_grpo_dataset(df, n=32, seed=SEED + 1)

print("Train:", grpo_train)
print("Eval :", grpo_eval)
print("\nVí dụ prompt:", grpo_train[0]["prompt"])

## 4. GRPO Training

In [ ]:
# Patch TRL import breakages
if "llm_blender" not in sys.modules:
    sys.modules["llm_blender"] = MagicMock()

import trl.extras.profiling as _trl_prof
if not hasattr(_trl_prof, "ProfilingContext"):
    _trl_prof.ProfilingContext = MagicMock()

print("TRL patches applied.")

In [ ]:
from trl import GRPOConfig, GRPOTrainer

grpo_args = GRPOConfig(
    output_dir=GRPO_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=4,
    max_completion_length=256,
    temperature=0.8,
    learning_rate=3e-6,
    beta=0.04,
    loss_type="grpo",
    gradient_checkpointing=True,
    bf16=torch.cuda.is_available(),
    logging_steps=1,
    save_steps=50,
    report_to="none",
)

grpo = GRPOTrainer(
    model=policy,
    args=grpo_args,
    processing_class=tok,
    reward_funcs=reward_fn,
    train_dataset=grpo_train,
    eval_dataset=grpo_eval,
)
grpo.train()
grpo.save_model(GRPO_DIR)
tok.save_pretrained(GRPO_DIR)
print("Đã lưu policy GRPO tại:", GRPO_DIR)

## 5. Thử sinh văn bản sau GRPO

In [ ]:
policy.eval()

test_prompt = extract_first_sentence(
    df.sample(1, random_state=SEED)["text"].values[0]
)
print("Prompt:", test_prompt)
print("-" * 60)

_dev = next(policy.parameters()).device
inputs = tok(test_prompt, return_tensors="pt")
inputs = {k: v.to(_dev) for k, v in inputs.items()}

out = policy.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_new_tokens=256,
    temperature=0.8,
    do_sample=True,
    pad_token_id=tok.pad_token_id,
)
generated = tok.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
print("Completion:")
print(generated)
print("-" * 60)

# Chấm điểm bằng reward model
rm_score = reward_fn([generated])[0]
print(f"Reward model score: {rm_score:.4f} (0=low engage, 1=high engage)")

## 6. LLM-as-a-Judge evaluation (sau train)

Dùng một LLM mạnh (GPT-4, Claude...) chấm điểm các completion để so sánh với reward model score.

**Hypothesis:** Reward model score cao (vì model học style của dataset) nhưng LLM judge score thấp (vì judge không quen với informal Vietnamese Facebook style).

In [ ]:
# Sinh nhiều completion để so sánh reward model vs LLM judge
policy.eval()
N_SAMPLES = 10

results = []
sample_prompts = [grpo_eval[i]["prompt"] for i in range(min(N_SAMPLES, len(grpo_eval)))]

for prompt in sample_prompts:
    inputs = tok(prompt, return_tensors="pt")
    inputs = {k: v.to(_dev) for k, v in inputs.items()}
    out = policy.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=200,
        temperature=0.8,
        do_sample=True,
        pad_token_id=tok.pad_token_id,
    )
    completion = tok.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    rm_score   = reward_fn([completion])[0]
    results.append({"prompt": prompt, "completion": completion, "rm_score": rm_score})

results_df = pd.DataFrame(results)
print(results_df[["rm_score", "completion"]].to_string())

In [ ]:
# --- LLM Judge prompt template ---
# Điền JUDGE_API_KEY và gọi API của LLM judge ở đây.
# Để trống để chạy thủ công hoặc tích hợp sau.

JUDGE_PROMPT_TEMPLATE = """\
Bạn là chuyên gia đánh giá nội dung marketing. \
Hãy chấm điểm bài đăng Facebook sau từ 1 đến 10 dựa trên: \
chất lượng nội dung, tính chuyên nghiệp, sự rõ ràng, và tiềm năng thu hút tương tác.

Bài đăng:
{completion}

Trả lời theo định dạng:
Score: <số>
Lý do: <giải thích ngắn>
"""

# Ví dụ gọi OpenAI (bỏ comment nếu có API key):
# import openai
# client = openai.OpenAI(api_key=os.environ["OPENAI_API_KEY"])
# 
# def llm_judge(completion):
#     resp = client.chat.completions.create(
#         model="gpt-4o",
#         messages=[{"role": "user", "content": JUDGE_PROMPT_TEMPLATE.format(completion=completion)}],
#         temperature=0,
#     )
#     return resp.choices[0].message.content
# 
# results_df["llm_judge_raw"] = results_df["completion"].apply(llm_judge)
# print(results_df[["rm_score", "llm_judge_raw"]].to_string())

print("Judge prompt template sẵn sàng — tích hợp API key để chạy.")